# DEMO_RUNBOOK — captured run record

Frozen, Markdown-only record of one complete pass through
[`DEMO_RUNBOOK.ipynb`](DEMO_RUNBOOK.ipynb), captured **2026-08-09** against a local API
process and the live CockroachDB Cloud cluster. It has no executable cells and no
connection details — the live notebook is the clean template to run yourself; this is
what a real pass through it produced.

Every block below is the run's own output, transcribed mechanically rather than retyped.

**Run identifiers** (all data synthetic, per ADR 005):

| | |
| --- | --- |
| `correlation_id` | `notebook-b545ccdc` |
| `incident_id` | `591be7cd-c17d-4172-989f-c7a409768342` |
| API version | `0.9.6` |
| Bedrock region | `eu-central-1` |
| Embedding model | `amazon.titan-embed-text-v2:0` |
| Reasoning model | `eu.anthropic.claude-sonnet-4-5-20250929-v1:0` |
| Runtime | `local` (local API process, not Lambda) |

`runtime` reads `local` because this capture drove the local API. It is taken from
Lambda's own `AWS_LAMBDA_FUNCTION_NAME` and is never defaulted, so a locally-run step
cannot claim Lambda — see the UI-badge rule in `CLAUDE.md`.

## 0. Preflight

`GET /api/v1/health`:

```json
{
  "status": "ok",
  "version": "0.9.6"
}
```

```
correlation_id for this session: notebook-b545ccdc
```

Live Bedrock was probed immediately before this run (`python scripts/probe_bedrock.py`):
every candidate region × both models returned OK. Quotas are dynamic, so that probe is
part of the capture, not a standing fact.

## 1. Fire a synthetic alert

The alert body is `DEMO_ALERT` from `scripts/demo_run.py` with this session's
`correlation_id` — identical to what sections 4 and 6 send, so the incident is opened
and resumed by the same alert:

```json
{
  "correlation_id": "notebook-b545ccdc",
  "service": "checkout-api",
  "region": "us-east-1",
  "severity": "high",
  "text": "[FIRING:1] HighLatencyP99 service=checkout-api region=us-east-1 severity=high — histogram_quantile(0.99, http_server_duration_seconds) = 2.41s exceeds SLO 0.80s for 5m; db_pool_connections_active 200/200, db_pool_clients_waiting 47"
}
```

Response:

```json
{
  "incident_id": "591be7cd-c17d-4172-989f-c7a409768342",
  "step_index": 0,
  "action": "scale_database_connection_pool",
  "state": "remediating",
  "resumed": false,
  "reexecuted_after_interrupt": false,
  "correlation_source": "bedrock",
  "reasoning_source": "bedrock"
}
```

`correlation_source` and `reasoning_source` both read `bedrock` — the live Titan
embedding and Claude reasoning paths ran. Both degrade silently by design, so these
markers are the only way to tell this from the precedent-replay fallback.

## 2. Read the live state back over MCP

`HTTP 200` from `/api/v1/incidents/open`, answered by the application
calling the CockroachDB Cloud Managed MCP Server's read-only SQL tool (ADR 003) — not by
a developer in an IDE.

```json
{
  "incidents": [
    {
      "incident_id": "591be7cd-c17d-4172-989f-c7a409768342",
      "service": "checkout-api",
      "state": "remediating",
      "opened_at": "2026-08-09T17:00:07.60246Z"
    },
    {
      "incident_id": "428f15ea-31b7-463d-92cb-ea412405994e",
      "service": "checkout-api",
      "state": "remediating",
      "opened_at": "2026-08-08T17:14:08.512007Z"
    },
    {
      "incident_id": "b440051e-47df-4274-b584-bec8f974a845",
      "service": "checkout-api",
      "state": "remediating",
      "opened_at": "2026-08-07T18:33:57.627862Z"
    },
    {
      "incident_id": "67740a62-416e-4f38-9769-2914211ad016",
      "service": "checkout-api",
      "state": "remediating",
      "opened_at": "2026-08-07T12:25:53.454661Z"
    },
    {
      "incident_id": "43595020-9d06-4237-b2da-1663566b156d",
      "service": "checkout-api",
      "state": "remediating",
      "opened_at": "2026-08-07T12:18:39.34369Z"
    },
    {
      "incident_id": "f7bc15e3-2042-4dea-b02b-7eeabb1bdbae",
      "service": "auth-service",
      "state": "remediating",
      "opened_at": "2026-07-07T17:46:35.956452Z"
    }
  ],
  "count": 6
}
```

This session's incident is first. The other five predate the capture and are deliberately
still open: four (`b440051e`, `67740a62`, `43595020`, `f7bc15e3`) are cited by
[`assets/provider-evidence/02.mcp-open-incidents.json`](../assets/provider-evidence/02.mcp-open-incidents.json),
so deleting them would leave committed evidence pointing at rows that no longer exist.
`428f15ea` is an earlier `demo-incident-001` tick from 2026-08-08. Abandoned debris from
this project's own notebook and debug runs was removed before this capture.

## 3. Advance the incident one step

Same alert, same `correlation_id`, posted again:

```json
{
  "incident_id": "591be7cd-c17d-4172-989f-c7a409768342",
  "step_index": 1,
  "action": "scale_database_connection_pool",
  "state": "remediating",
  "resumed": true,
  "reexecuted_after_interrupt": false,
  "correlation_source": "bedrock",
  "reasoning_source": "bedrock"
}
```

```
same incident as section 1? True
```

`incident_id` unchanged, `step_index` advanced
0 → 1, `resumed: true` — the
orchestrator read the open incident out of CockroachDB and continued it rather than
opening a second one.

## 4. The kill — real `TerminateProcess`, mid-step

```bash
python scripts/demo_run.py --tick --via-api --correlation-id notebook-b545ccdc
python scripts/chaos_kill.py --port 8000
```

The strike was timed by polling CockroachDB until step 2's `executing` checkpoint
was durable, rather than by sleeping a guessed interval — too early and the kill lands
before anything durable exists, which is the failure mode the live notebook warns about.

Step 2 committed `executing` at **17:01:25**; the kill landed at
**17:01:27** — 2s later, inside the 5s `step_execution_seconds` window:

```json
{
  "pid": 47132,
  "name": "python.exe",
  "port": 8000,
  "note": "no graceful shutdown — hard kill",
  "event": "killing_process",
  "timestamp": "2026-08-09T17:01:27.177077Z",
  "level": "info"
}
```

No graceful shutdown, no checkpoint call, no cleanup hook. The in-flight client saw a
severed connection rather than a clean error:

```
httpx.ReadError: [WinError 10054] An existing connection was forcibly closed by the remote host
```

## 5. Confirm the state outlived the process

Queried CockroachDB directly, with the API process dead and nothing else running:

```
incident 591be7cd-c17d-4172-989f-c7a409768342

17:00:14  step 0  executed   scale_database_connection_pool
17:00:35  step 1  executed   scale_database_connection_pool
17:01:25  step 2  executing  increase_database_connection_pool_size  <-- died here
```

Step 2 sits durably in `executing` — committed by `checkpoint_step_start` before the
execution window (ADR 009), with no process alive to own it. **This is the thesis, and the
shot the demo video leads with.**

The durable `detail` on that row, read straight back out of the database:

```json
{
  "based_on": "7ad8682a-e21a-4e4e-a068-da84d0346f5a",
  "bedrock_region": "eu-central-1",
  "correlation_source": "bedrock",
  "embedding_model_id": "amazon.titan-embed-text-v2:0",
  "precedent_distance": 0.790228,
  "precedent_rank": 0,
  "precedent_state": "resolved",
  "precedent_summary": "Elevated p99 latency on checkout-api in us-east-1, likely connection pool exhaustion",
  "precedents_considered": 5,
  "rationale": "Database connection pool is completely exhausted (200/200 active, 47 waiting) which directly correlates with the elevated p99 latency, matching the closest past incident pattern of connection pool exhaustion.",
  "reasoning_model_id": "eu.anthropic.claude-sonnet-4-5-20250929-v1:0",
  "reasoning_source": "bedrock",
  "reexecuted_after_interrupt": true,
  "runtime": "local"
}
```

That JSONB is self-describing on purpose: the row records how it was reasoned about, so
evidence captured after the fact needs no external notes. `precedent_distance`
0.790228 at rank 0 independently reproduces the
≈0.7902 retrieval measured against the committed Titan fixture in `scripts/demo_run.py` —
and it renders alongside the `embedding_model_id` that produced it, because distances are
not comparable across embedding models.

## 6. The recovery

A genuinely cold `python -m uvicorn api.main:app --port 8000` process — the previous one
was killed, not restarted — then the same alert on the same `correlation_id`:

```bash
python -m uvicorn api.main:app --port 8000
python scripts/demo_run.py --tick --via-api --resume-check --correlation-id notebook-b545ccdc
```

```json
{
  "incident_id": "591be7cd-c17d-4172-989f-c7a409768342",
  "step_index": 2,
  "action": "increase_database_connection_pool_size",
  "state": "resolved",
  "resumed": true,
  "reexecuted_after_interrupt": true,
  "correlation_source": "bedrock",
  "reasoning_source": "bedrock"
}
```

`reexecuted_after_interrupt: true` — the cold process read the durable `executing` row and
re-ran exactly that step, then resolved the incident (step 2 was the last of
`max_remediation_steps`). Re-querying CockroachDB:

```
incident 591be7cd-c17d-4172-989f-c7a409768342

17:00:14  step 0  executed   scale_database_connection_pool
17:00:35  step 1  executed   scale_database_connection_pool
17:01:25  step 2  executed   increase_database_connection_pool_size
```

**Step rows before the resume: 3. After: 3.** The interrupted
step advanced in place from `executing` to `executed` — not skipped, not duplicated. The row
count is the exactly-once proof; the status transition is the re-execution proof.

Incident state is now `resolved`.

In this run the re-proposed action matched the durable one. It need not: the resume path
updates `status` and merges `detail` but never rewrites `action`, so the durable row always
keeps what the killed invocation committed while the response reports what this pass
re-proposed. Claude's re-proposal is not deterministic, so the two sometimes differ — which
is why the row count, not the action string, is what the exactly-once claim rests on.

---

### What this run proves

- A step killed mid-execution stayed durably `executing` in CockroachDB with the owning
  process dead — 17:01:25 checkpoint, 17:01:27 kill (2s later).
- A cold-started process read that row and re-executed the interrupted step **exactly
  once** — 3 step rows before the resume, 3 after — then
  resolved the incident.
- The kill and the resume targeted the same incident the notebook itself opened, driven by
  the same alert body, via `scripts/demo_run.py --correlation-id`.
- Both Bedrock paths ran for real throughout (`correlation_source` / `reasoning_source`:
  `bedrock`), with the models and precedent distance recorded durably on the step itself —
  not inferred from which code path exists.

### What this run does not show

- **Lambda.** `runtime` reads `local`; this capture drove the local API process. The same
  guarantee on the deployed function is evidenced separately under
  [`assets/deploy-restart-run/`](../assets/deploy-restart-run/).
- **Concurrent invocations.** The exactly-once claim here is one killed step resumed once.
  The racing-invocation case is asserted against a real cluster in
  [`tests/integration/test_recovery_e2e.py`](../tests/integration/test_recovery_e2e.py),
  and the literal process-kill in
  [`tests/integration/test_chaos_kill_e2e.py`](../tests/integration/test_chaos_kill_e2e.py).